In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df1=pd.read_csv('/content/data.csv')

df=df1.copy()

#first 5 rows
df.head()

#checkdtypes 
df.info()

#change dtype for date
df["DATE"]=pd.to_datetime(df["DATE"],errors='coerce',infer_datetime_format=True)

#change_month
df['MONTH_new'] =df["DATE"].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['MONTH_new']/12)
df['month_cos'] = np.cos(2 * np.pi * df['MONTH_new']/12)


print(df.duplicated().sum())



#check for correlation

corr_matrix = df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap: Weather Features vs Visibility")
plt.show()

#corr between drybulbtemp and wetbulbtemp
#corr between dewpoint temp and dry bulbtemp
#corr betwwn dewpoint temp and wetbulbtemp
#corr between stationlevel and sea level pressure



# (How close the air is to saturation/fog)
df['DewPointDepression'] = df['DRYBULBTEMPF'] - df['DewPointTempF']

#Measures evaporative capacity
df['WetBulbDepression'] = df['DRYBULBTEMPF'] - df['WETBULBTEMPF']


#clip negative values to 0
df['DewPointDepression'] = df['DewPointDepression'].clip(lower=0)
df['WetBulbDepression'] = df['WetBulbDepression'].clip(lower=0)


df = df.drop(columns=['WETBULBTEMPF', 'DewPointTempF',"SeaLevelPressure"])


corr_matrix = df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap: Weather Features vs Visibility")
plt.show()

df= df.drop(columns=['RelativeHumidity', 'DewPointDepression'])


df.head()

corr_matrix = df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap: Weather Features vs Visibility")
plt.show()


#outlier detection using boxplot
numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
plt.figure(figsize=(15, 5 * n_rows))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(f'Boxplot of {col}', fontsize=12)
    plt.ylabel('Value')

plt.tight_layout()
plt.savefig('numeric_boxplots.png')
plt.show()

#clipping to resolve outliers
cols_to_cap = ['WindSpeed', 'Precip', 'WetBulbDepression']

for col in cols_to_cap:
    upper_limit = df[col].quantile(0.80)
    lower_limit = df[col].quantile(0.20)
    df[col] = df[col].clip(upper=upper_limit,lower=lower_limit)
df["StationPressure"]=df["StationPressure"].clip(upper=df["StationPressure"].quantile(0.90),lower=df["StationPressure"].quantile(0.1))


#introducing hour column
df['hour'] = df['DATE'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['hour_sin'] = df['hour_sin'].round(4)
df['hour_cos'] = df['hour_cos'].round(4)

#CHECKING VARIATION INFLATION FACTOR
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(considered_features, df):
  X = df[considered_features].copy().select_dtypes(include=['number'])
  X['intercept'] = 1
  vif = pd.DataFrame()
  vif["variables"] = X.columns
  vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
  return vif[vif['variables'] != 'intercept'].sort_values('VIF', ascending=False)


features = df.select_dtypes(include=['number']).columns.to_list()
if 'Visibility' in features:
    features.remove('Visibility')

vif_results = compute_vif(features, df)
print(vif_results)

#Every single one of your features is below 7.0.

#outlier detection using boxplot
numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
plt.figure(figsize=(15, 5 * n_rows))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(f'Boxplot of {col}', fontsize=12)
    plt.ylabel('Value')

plt.tight_layout()
plt.savefig('numeric_boxplots.png')
plt.show()


import math



plot_df = df.drop(['DATE'], axis=1).select_dtypes(include=['number'])
cols = plot_df.columns
n_cols = 3
n_rows = math.ceil(len(cols) / n_cols)

plt.figure(figsize=(20, 5 * n_rows), facecolor='white')

for i, column in enumerate(cols, 1):
    ax = plt.subplot(n_rows, n_cols, i)
    sns.kdeplot(plot_df[column], fill=True, color='skyblue') # Added fill for better visibility
    plt.xlabel(column, fontsize=15)
    plt.title(f"Distribution of {column}", fontsize=12)

plt.tight_layout()
plt.savefig('multivariate_analysis.png')
plt.show()

#some features dont follow gaussian distribution
#we will use models that dont require normal distribution

#implementing changes
df = df.sort_values('DATE')

df['Visibility_Lag1'] = df['VISIBILITY'].shift(1)
df['Visibility_Trend'] = df['VISIBILITY'].shift(1) - df['VISIBILITY'].shift(2)
df = df.dropna(subset=['Visibility_Lag1', 'Visibility_Trend'])

#separating features and target 
df=df.drop(columns=['DATE', 'MONTH_new', 'hour','Precip'])

#clustering

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


necessary_features = [
    'DRYBULBTEMPF', 
    'RelativeHumidity', 
    'WindSpeed', 
    'StationPressure', 
    'month_sin', 'month_cos', 
    'hour_sin', 'hour_cos',
    'DewPointDepression', 
    'WetBulbDepression',
    
]

X = df[necessary_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

best_k = 2
best_score = -1
scores = []
k_range = range(2, 6)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled,labels)
    scores.append(score)
    print(f"k={k} | Silhouette Score: {score:.4f}")
    if score > best_score:
        best_score = score
        best_k = k

print(f"\nWinner: Optimal Clusters (k) = {best_k} with score {best_score:.4f}")

#fit the 
final_kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
train_clusters = final_kmeans.fit_predict(X_scaled)

labels=final_kmeans.labels_
df['Cluster'] = labels
print(df['Cluster'].value_counts())
df.head()

!pip install dask[distributed] --upgrade

import pandas as pd
import numpy as np
import joblib
import dask
from dask.distributed import Client, LocalCluster 
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


try:
    client = Client() 
    print(f"Dask Dashboard link: {client.dashboard_link}")
except Exception as e:
    print(f"Dask Error: {e}")

models_to_test = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'RandomForest': RandomForestRegressor(n_estimators=100),
    'XGBoost': XGBRegressor()
}


def train_and_validate_by_cluster(dataframe, models_dict):
    results_report = []

    for cluster_id in sorted(df['Cluster'].unique()):
        print(f" Training Models for Cluster {cluster_id}")
        c_data = dataframe[dataframe['Cluster'] == cluster_id]
        X_c = c_data.drop(columns=['VISIBILITY', 'Cluster'], errors='ignore').select_dtypes(include=[np.number])
        y_c = c_data['VISIBILITY']

        x_train, x_test, y_train, y_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

        with joblib.parallel_backend('dask'):
            for name, model in models_dict.items():
                model.fit(x_train, y_train)
                y_pred = model.predict(x_test)
                score = r2_score(y_test, y_pred)
                
                results_report.append({
                    'Cluster': cluster_id,
                    'Model': name,
                    'R2_Score': score
                })
                print(f"Cluster {cluster_id} | {name} R2: {score:.4f}")
    


    return pd.DataFrame(results_report)

   
                
comparison_df = train_and_validate_by_cluster(df, models_to_test)


#random forest is better


import joblib
from sklearn.model_selection import GridSearchCV
import os

joblib.dump(final_kmeans, 'models/final_kmeans.pkl')

rf_grid = {
    'n_estimators': [50, 100, 200],      # Number of trees
    'max_depth': [10, 20, None],         # Depth (None = grow until pure)
    'min_samples_split': [2, 5],         # Minimum data points to split a node
    'max_features': ['sqrt', 'log2']     # Number of features to consider per split
}

os.makedirs('models/final_models', exist_ok=True)

final_experts = {}

for cluster_id in sorted(df['Cluster'].unique()):
    
    print(f"FINALIZE: Tuning Champion for Cluster {cluster_id}")
   
    c_data = df[df['Cluster'] == cluster_id]
    X_c = c_data.drop(columns=['VISIBILITY', 'Cluster'], errors='ignore').select_dtypes(include=[np.number])
    y_c = c_data['VISIBILITY']

    X_train, X_test, y_train, y_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

    cluster_scaler = StandardScaler()
    X_train_scaled = cluster_scaler.fit_transform(X_train)
    X_test_scaled = cluster_scaler.transform(X_test)

    rf_base = RandomForestRegressor(random_state=42)
    grid_search = GridSearchCV(
        estimator=rf_base, 
        param_grid=rf_grid, 
        cv=3, 
        scoring='r2', 
        n_jobs=-1
    )
    
    
    with joblib.parallel_backend('dask'):
        print(f"Testing {len(rf_grid['n_estimators']) * len(rf_grid['max_depth']) * 4} combinations...")
        grid_search.fit(X_train_scaled,y_train)
        
    print(f"Best Parameters: {grid_search.best_params_}")
    best_model = grid_search.best_estimator_
    joblib.dump(best_model, f'models/experts/rf_expert_cluster_{cluster_id}.pkl')
    joblib.dump(cluster_scaler, f'models/experts/scaler_cluster_{cluster_id}.pkl')

    final_experts[cluster_id] = {
        'R2_Score': round(grid_search.best_score_, 4),
        'Best_Params': grid_search.best_params_
    }
    
    print(f"Cluster {cluster_id} complete!")
    print(f"Best R2: {grid_search.best_score_:.4f}")
    print(f"Params: {grid_search.best_params_}")

    print(f"Cluster {cluster_id} complete! Test R2: {final_experts[cluster_id]['R2_Score']}")

print("ALL EXPERTS TRAINED AND SAVED ")


#shap

import shap

# 1. Load your Expert and Scaler
expert_1 = joblib.load('models/experts/rf_expert_cluster_2.pkl')
scaler_1 = joblib.load('models/experts/scaler_cluster_2.pkl')

c_data = df[df['Cluster'] == 2]


X_sample_raw= c_data.drop(columns=['VISIBILITY', 'Cluster'], errors='ignore').select_dtypes(include=[np.number]).head(100) 


X_sample_scaled = scaler_1.transform(X_sample_raw)

explainer = shap.TreeExplainer(expert_1)
shap_values = explainer.shap_values(X_sample_scaled)


shap.summary_plot(shap_values, X_sample_scaled, feature_names=[
    'DRYBULBTEMPF', 
    'RelativeHumidity', 
    'WindSpeed', 
    'StationPressure', 
    'month_sin', 'month_cos', 
    'hour_sin', 'hour_cos',
    'DewPointDepression', 
    'WetBulbDepression','Visibility_Lag1', 'Visibility_Trend','WindDirection'
    
])

plt.savefig('cluster_2_shap_analysis.png', bbox_inches='tight', dpi=300)



